# 🎙️ Turkish Voice Assistant v2.0

**Whisper Small TR** + **LM Studio** + **Tool Registry** + **Cache** + **Memory**

---

## 🏗️ Architecture

```
┌─────────────┐     ┌─────────────┐     ┌─────────────┐
│   Whisper   │ ──→ │    Agent    │ ──→ │   Gradio    │
│    (ASR)    │     │  + Memory   │     │    (UI)     │
└─────────────┘     └──────┬──────┘     └─────────────┘
                          │
         ┌────────────────┼────────────────┐
         ↓                ↓                ↓
   ┌──────────┐    ┌──────────┐    ┌──────────┐
   │  Cache   │    │   LLM    │    │  Tools   │
   │  (Dict)  │    │ (ngrok)  │    │(Registry)│
   └──────────┘    └──────────┘    └──────────┘
```

## 🔧 Setup

```bash
# Open LM Studio, load model, start server
ngrok http 1234
```

## 1️⃣ Libraries

In [1]:
#!pip install -q transformers torch accelerate
#!pip install -q librosa soundfile
#!pip install -q ddgs
#!pip install -q gradio
#!pip install -q openai

## 2️⃣ Configuration & NGROK URL

In [2]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  ⚙️  ALL CONFIGURATION - IN ONE PLACE!                                       ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

import os

CONFIG = {
    # 🔗 LM Studio connection
    # Get from environment variable or use placeholder
    # For production, set NGROK_URL environment variable
    "ngrok_url": os.getenv("NGROK_URL", "http://localhost:1234"),  # <-- CHANGE THIS!
    
    # 🎤 Whisper settings
    "whisper_model": "emredeveloper/whisper-small-tr",
    "chunk_length": 30,
    
    # 🤖 LLM settings
    "tool_temperature": 0.3,    # For tool selection (low = consistent)
    "summary_temperature": 0.5, # For summary (medium = creative but consistent)
    
    # 💾 Cache settings
    "cache_enabled": True,
    "cache_ttl": 300,  # 5 minutes (seconds)
    
    # 🧠 Memory settings
    "memory_enabled": True,
    "memory_size": 10,  # Last 10 messages
    
    # 🔄 Retry settings
    "max_retries": 3,
    "retry_delay": 1,  # seconds
}

NGROK_URL = CONFIG["ngrok_url"]
print(f"🔗 LM Studio: {NGROK_URL}")
print(f"💾 Cache: {'✅' if CONFIG['cache_enabled'] else '❌'}")
print(f"🧠 Memory: {'✅' if CONFIG['memory_enabled'] else '❌'} ({CONFIG['memory_size']} messages)")

🔗 LM Studio: https://ascocarpous-nonvariously-porsha.ngrok-free.dev
💾 Cache: ✅
🧠 Memory: ✅ (10 mesaj)


## 3️⃣ Setup & Connection

In [3]:
import torch
import json
import time
import hashlib
from datetime import datetime, timezone
from functools import wraps
from openai import OpenAI
import warnings
warnings.filterwarnings("ignore")

# ═══════════════════════════════════════════════════════════════════════════════
# 🔧 HELPER TOOLS
# ═══════════════════════════════════════════════════════════════════════════════

# 📝 Logger
class Logger:
    @staticmethod
    def info(msg): print(f"ℹ️  {msg}")
    @staticmethod
    def success(msg): print(f"✅ {msg}")
    @staticmethod
    def error(msg): print(f"❌ {msg}")
    @staticmethod
    def tool(msg): print(f"🔧 {msg}")
    @staticmethod
    def cache(msg): print(f"💾 {msg}")

log = Logger()

# 💾 Cache
class Cache:
    def __init__(self):
        self._store = {}
    
    def _hash(self, key):
        return hashlib.md5(key.encode()).hexdigest()
    
    def get(self, key):
        if not CONFIG["cache_enabled"]:
            return None
        h = self._hash(key)
        if h in self._store:
            val, ts = self._store[h]
            if time.time() - ts < CONFIG["cache_ttl"]:
                log.cache(f"HIT: {key[:30]}...")
                return val
            del self._store[h]
        return None
    
    def set(self, key, value):
        if CONFIG["cache_enabled"]:
            self._store[self._hash(key)] = (value, time.time())
    
    def clear(self):
        self._store.clear()
        log.cache("Cleared")

cache = Cache()

# 🔄 Retry decorator
def retry(max_tries=None, delay=None):
    max_tries = max_tries or CONFIG["max_retries"]
    delay = delay or CONFIG["retry_delay"]
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(max_tries):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    if attempt < max_tries - 1:
                        log.info(f"Retry {attempt+1}/{max_tries}: {e}")
                        time.sleep(delay)
                    else:
                        raise e
        return wrapper
    return decorator

# ═══════════════════════════════════════════════════════════════════════════════
# 🔌 CONNECTION
# ═══════════════════════════════════════════════════════════════════════════════

WHISPER_MODEL = CONFIG["whisper_model"]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

client = OpenAI(base_url=f"{NGROK_URL}/v1", api_key="not-needed")

log.info(f"Device: {DEVICE}")
log.info(f"Whisper: {WHISPER_MODEL}")

try:
    models = client.models.list()
    MODEL_ID = models.data[0].id if models.data else "default"
    log.success(f"Connection successful! Model: {MODEL_ID}")
except Exception as e:
    log.error(f"Connection error: {e}")

ℹ️  Cihaz: cuda
ℹ️  Whisper: emredeveloper/whisper-small-tr
✅ Bağlantı başarılı! Model: qwen/qwen3-vl-4b


## 4️⃣ Tool Definitions

In [4]:
from ddgs import DDGS
import requests

# ═══════════════════════════════════════════════════════════════════════════════
# 📦 TOOL REGISTRY - Just use @register_tool to add new tools!
# ═══════════════════════════════════════════════════════════════════════════════

class ToolRegistry:
    def __init__(self):
        self._tools = {}
    
    def register(self, name: str, description: str, parameters: dict = None):
        """Decorator: Register function as a tool"""
        def decorator(func):
            self._tools[name] = {
                "func": func,
                "schema": {
                    "type": "function",
                    "function": {
                        "name": name,
                        "description": description,
                        "parameters": parameters or {"type": "object", "properties": {}, "required": []}
                    }
                }
            }
            return func
        return decorator
    
    @property
    def schemas(self) -> list:
        """OpenAI format tool schemas"""
        return [t["schema"] for t in self._tools.values()]
    
    @property
    def functions(self) -> dict:
        """Tool functions map"""
        return {k: v["func"] for k, v in self._tools.items()}
    
    def call(self, name: str, **kwargs):
        """Call tool"""
        if name in self._tools:
            return self._tools[name]["func"](**kwargs)
        return f"Unknown tool: {name}"
    
    def list(self) -> list:
        return list(self._tools.keys())

registry = ToolRegistry()

# ═══════════════════════════════════════════════════════════════════════════════
# 🔧 TOOL DEFINITIONS - Add new tools here!
# ═══════════════════════════════════════════════════════════════════════════════

@registry.register(
    name="web_search",
    description="İnternette arama yapar. Bilgi, tanım, haber veya güncel konu sorularında kullan.",
    parameters={
        "type": "object",
        "properties": {"query": {"type": "string", "description": "Aranacak konu"}},
        "required": ["query"]
    }
)
def web_search(query: str) -> str:
    # Cache check
    cached = cache.get(f"search:{query}")
    if cached:
        return cached
    
    try:
        results = []
        for r in DDGS().text(query, max_results=5):
            results.append(f"- {r.get('title', '')}: {r.get('body', '')}")
        result = "\n".join(results) if results else "No results found."
        cache.set(f"search:{query}", result)
        return result
    except Exception as e:
        return f"Error: {e}"


@registry.register(
    name="calculate",
    description="Matematiksel hesaplama yapar. Toplama, çıkarma, çarpma, bölme işlemlerinde kullan.",
    parameters={
        "type": "object",
        "properties": {"expression": {"type": "string", "description": "Hesaplanacak ifade (örn: 125*48)"}},
        "required": ["expression"]
    }
)
def calculate(expression: str) -> str:
    try:
        expr = expression.replace("×", "*").replace("÷", "/").replace("x", "*")
        result = eval(expr)
        return str(result)
    except:
        return f"Could not calculate: {expression}"


@registry.register(
    name="get_current_time",
    description="Şu anki tarih ve saati döndürür. Saat kaç, bugün ne gün sorularında kullan."
)
def get_current_time() -> str:
    now = datetime.now(timezone.utc)
    return f"{now.strftime('%d %B %Y, %H:%M:%S')} UTC"


@registry.register(
    name="get_weather",
    description="Hava durumu bilgisi verir. Şehir adı gerekli.",
    parameters={
        "type": "object",
        "properties": {"city": {"type": "string", "description": "Şehir adı (örn: Istanbul)"}},
        "required": ["city"]
    }
)
def get_weather(city: str) -> str:
    # Cache check
    cached = cache.get(f"weather:{city}")
    if cached:
        return cached
    
    # Normalize city name
    city_clean = city.strip().replace(" ", "+")
    
    # Try multiple sources (fallback) - m=metric (Celsius)
    sources = [
        f"https://wttr.in/{city_clean}?format=%l:+%c+%t+%h+%w&lang=tr&m",
        f"https://wttr.in/{city_clean}?format=3&lang=tr&m",
        f"https://wttr.in/{city_clean}?format=%C+%t&lang=tr&m",
    ]
    
    for url in sources:
        for attempt in range(2):  # 2 attempts per source
            try:
                r = requests.get(url, timeout=10, headers={"User-Agent": "curl/7.68.0"})
                if r.status_code == 200 and r.text.strip() and "Unknown" not in r.text:
                    result = r.text.strip()
                    cache.set(f"weather:{city}", result)
                    return result
            except:
                time.sleep(0.5)
                continue
    
    # If none work, try web search
    try:
        search_result = web_search(f"{city} hava durumu bugün")
        if search_result and "Error" not in search_result:
            return f"🌤️ {city} hava durumu:\n{search_result[:300]}"
    except:
        pass
    
    return f"⚠️ Weather for {city} cannot be retrieved at the moment. Please try again."


@registry.register(
    name="ask_clarification",
    description="Kullanıcıdan netleştirme ister. Anlamsız, belirsiz veya eksik sorularda MUTLAKA kullan!",
    parameters={
        "type": "object",
        "properties": {"reason": {"type": "string", "description": "Neden netleştirme gerektiğini açıkla"}},
        "required": ["reason"]
    }
)
def ask_clarification(reason: str) -> str:
    return f"CLARIFY:{reason}"


# For backward compatibility
TOOLS = registry.schemas
TOOL_MAP = registry.functions

log.success(f"Tools: {', '.join(registry.list())}")

✅ Tool'lar: web_search, calculate, get_current_time, get_weather, ask_clarification


## 5️⃣ Load Whisper

In [5]:
from transformers import pipeline

log.info("Loading Whisper...")
whisper_pipe = pipeline(
    "automatic-speech-recognition",
    model=WHISPER_MODEL,
    chunk_length_s=CONFIG["chunk_length"],
    device=DEVICE,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
)
log.success("Whisper ready!")

ℹ️  Whisper yükleniyor...


`torch_dtype` is deprecated! Use `dtype` instead!
Device set to use cuda
Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


✅ Whisper hazır!


## 6️⃣ Agent (LLM Auto Tool Selection)

In [6]:
class Agent:
    """
    LLM selects tools and summarizes results.
    Maintains context with memory.
    """
    
    def __init__(self):
        self.memory = []  # Conversation history
        
        self.tool_system = """Sen bir asistansın. Kullanıcının sorusunu cevaplamak için uygun aracı seç.
- Bilgi/tanım/haber → web_search
- Hesaplama → calculate
- Saat/tarih → get_current_time
- Hava durumu → get_weather
- Anlamsız/belirsiz → ask_clarification"""

        self.summary_system = """Araç sonucunu kullanarak soruya kısa ve öz Türkçe cevap ver.

KURALLAR:
1. Sadece düz metin yaz, XML/JSON/kod yazma
2. Sonuçtaki bilgileri DEĞİŞTİRME - aynen aktar
3. Tarih/süre bilgilerini DOĞRU aktar (52 hafta ≠ 1 hafta!)
4. Emin değilsen "Sonuçlara göre..." diye başla
5. Yanlış bilgi vermektense "bu bilgi bulunamadı" de"""
    
    def _add_memory(self, role: str, content: str):
        """Add message to memory"""
        if CONFIG["memory_enabled"]:
            self.memory.append({"role": role, "content": content})
            # Size control
            if len(self.memory) > CONFIG["memory_size"] * 2:
                self.memory = self.memory[-CONFIG["memory_size"] * 2:]
    
    def _get_context(self) -> str:
        """Get recent conversations as context"""
        if not CONFIG["memory_enabled"] or not self.memory:
            return ""
        recent = self.memory[-CONFIG["memory_size"]:]
        ctx = "\n".join([f"{m['role']}: {m['content'][:100]}" for m in recent])
        return f"\n\nPrevious conversation:\n{ctx}"
    
    def clear_memory(self):
        """Clear memory"""
        self.memory.clear()
        log.info("Memory cleared")
    
    @retry()
    def _call_llm(self, messages, tools=None, tool_choice=None, temperature=0.5):
        """LLM call (with retry)"""
        kwargs = {"model": MODEL_ID, "messages": messages, "temperature": temperature}
        if tools:
            kwargs["tools"] = tools
            kwargs["tool_choice"] = tool_choice or "auto"
        return client.chat.completions.create(**kwargs)
    
    def chat(self, message: str) -> tuple:
        tool_info = None
        context = self._get_context()
        
        try:
            # 1. Tool selection
            response = self._call_llm(
                messages=[
                    {"role": "system", "content": self.tool_system + context},
                    {"role": "user", "content": message}
                ],
                tools=TOOLS,
                tool_choice="required",
                temperature=CONFIG["tool_temperature"]
            )
            
            msg = response.choices[0].message
            
            if not msg.tool_calls:
                return "❓ I couldn't understand. Please try again.", {"needs_input": True}
            
            tool_call = msg.tool_calls[0]
            func_name = tool_call.function.name
            func_args = json.loads(tool_call.function.arguments)
            
            log.tool(f"{func_name}({func_args})")
            
            # 2. Execute tool
            result = registry.call(func_name, **func_args)
            
            # Clarification?
            if func_name == "ask_clarification":
                reason = func_args.get("reason", "Unclear question")
                return f"❓ {reason}\n\n💬 Please clarify:", {
                    "name": func_name, "args": func_args, "needs_input": True
                }
            
            log.info(f"Result: {str(result)[:100]}...")
            
            tool_info = {"name": func_name, "args": func_args, "result": str(result)[:500]}
            
            # 3. Summarize result
            summary = self._call_llm(
                messages=[
                    {"role": "system", "content": self.summary_system},
                    {"role": "user", "content": f"Soru: {message}\n\nAraç sonucu:\n{result}"}
                ],
                temperature=CONFIG["summary_temperature"]
            )
            
            answer = summary.choices[0].message.content
            
            # Clean if contains XML/tool_call
            if "<tool_call>" in answer or "```" in answer:
                answer = f"📊 {func_name} sonucu:\n{str(result)[:500]}"
            
            # Add to memory
            self._add_memory("user", message)
            self._add_memory("assistant", answer[:200])
            
            return answer, tool_info
            
        except Exception as e:
            log.error(f"Agent error: {e}")
            return f"Error: {e}", None


agent = Agent()
log.success("Agent ready! (Memory + Cache + Retry)")

✅ Agent hazır! (Memory + Cache + Retry)


## 7️⃣ Voice Assistant

In [7]:
class VoiceAgent:
    @retry()
    def transcribe(self, audio):
        if not audio:
            return ""
        try:
            result = whisper_pipe(audio)["text"].strip()
            log.info(f"Transcript: {result}")
            return result
        except Exception as e:
            log.error(f"Transcription error: {e}")
            return ""
    
    def process_audio(self, audio):
        text = self.transcribe(audio)
        if not text:
            return "", "Audio could not be understood.", None
        resp, tool = agent.chat(text)
        return text, resp, tool
    
    def process_text(self, text):
        if not text.strip():
            return "Please enter a message.", None
        return agent.chat(text)

voice = VoiceAgent()
log.success("Voice assistant ready!")

✅ Sesli asistan hazır!


## 8️⃣ Gradio UI

In [8]:
import gradio as gr

# ═══════════════════════════════════════════════════════════════════════════════
# 🎨 UI HELPERS
# ═══════════════════════════════════════════════════════════════════════════════

def get_status():
    mem = len(agent.memory) if CONFIG["memory_enabled"] else 0
    cch = len(cache._store) if CONFIG["cache_enabled"] else 0
    return f"🧠 {mem} | 💾 {cch}"

def format_tool(tool: dict) -> str:
    if not tool:
        return '<div class="tool-empty">Waiting for tool...</div>'
    
    icons = {"web_search": "🔍", "calculate": "🔢", "get_current_time": "🕐", "get_weather": "🌤️", "ask_clarification": "❓"}
    name = tool.get("name", "?")
    icon = icons.get(name, "🔧")
    args = tool.get("args", {})
    result = str(tool.get("result", ""))[:300]
    args_str = ", ".join([f"<b>{k}</b>={v}" for k, v in args.items()])
    
    return f'''<div class="tool-card">
        <div class="tool-header">{icon} {name}</div>
        <div class="tool-args">{args_str}</div>
        <div class="tool-result">{result}</div>
    </div>'''

def process_audio_ui(audio):
    if not audio:
        return "⚠️ Ses yükleyin", "", '<div class="tool-empty">-</div>', get_status()
    text, resp, tool = voice.process_audio(audio)
    return text or "?", resp or "", format_tool(tool), get_status()

def process_text_ui(text_input):
    if not text_input.strip():
        return "", "💬 Mesaj yazın", '<div class="tool-empty">-</div>', get_status()
    resp, tool = voice.process_text(text_input)
    return text_input, resp or "", format_tool(tool), get_status()

def clear_all():
    agent.clear_memory()
    cache.clear()
    return "", "", '<div class="tool-empty">Cleared ✓</div>', get_status()

def run_example(q):
    resp, tool = voice.process_text(q)
    return q, resp or "", format_tool(tool), get_status()

# ═══════════════════════════════════════════════════════════════════════════════
# 🎨 CUSTOM CSS - Modern Glassmorphism
# ═══════════════════════════════════════════════════════════════════════════════

css = """
/* Main Container - FULL WIDTH */
.gradio-container {
    max-width: 100% !important;
    width: 100% !important;
    padding: 20px !important;
    background: linear-gradient(135deg, #0f0c29 0%, #302b63 50%, #24243e 100%) !important;
    min-height: 100vh !important;
}

/* Header */
.header-title {
    text-align: center;
    background: linear-gradient(90deg, #667eea 0%, #764ba2 100%);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
    font-size: 2.5rem !important;
    font-weight: 800 !important;
    margin-bottom: 0.5rem !important;
}
.header-sub {
    text-align: center;
    color: #a0a0a0;
    font-size: 0.9rem;
}

/* Cards */
.input-card, .output-card {
    background: rgba(255,255,255,0.05) !important;
    backdrop-filter: blur(10px) !important;
    border: 1px solid rgba(255,255,255,0.1) !important;
    border-radius: 16px !important;
    padding: 20px !important;
}

/* Transcription */
.trans-box textarea {
    background: rgba(102, 126, 234, 0.1) !important;
    border: 1px solid rgba(102, 126, 234, 0.3) !important;
    border-radius: 12px !important;
    font-size: 1.1rem !important;
    color: #e0e0e0 !important;
    padding: 16px !important;
}

/* Response */
.resp-box textarea {
    background: linear-gradient(135deg, rgba(118,75,162,0.15) 0%, rgba(102,126,234,0.15) 100%) !important;
    border: 1px solid rgba(118,75,162,0.3) !important;
    border-radius: 12px !important;
    font-size: 1.2rem !important;
    line-height: 1.8 !important;
    color: #ffffff !important;
    padding: 20px !important;
    min-height: 150px !important;
}

/* Tool Card */
.tool-card {
    background: linear-gradient(135deg, rgba(0,200,150,0.1) 0%, rgba(0,150,200,0.1) 100%);
    border: 1px solid rgba(0,200,150,0.3);
    border-radius: 12px;
    padding: 16px;
    margin-top: 8px;
}
.tool-header {
    font-size: 1.1rem;
    font-weight: 700;
    color: #00c896;
    margin-bottom: 8px;
}
.tool-args {
    font-size: 0.85rem;
    color: #a0d0ff;
    margin-bottom: 8px;
    font-family: monospace;
}
.tool-result {
    font-size: 0.9rem;
    color: #d0d0d0;
    background: rgba(0,0,0,0.2);
    padding: 12px;
    border-radius: 8px;
    white-space: pre-wrap;
    max-height: 120px;
    overflow-y: auto;
}
.tool-empty {
    color: #606060;
    font-style: italic;
    padding: 20px;
    text-align: center;
}

/* Buttons */
.primary-btn {
    background: linear-gradient(90deg, #667eea 0%, #764ba2 100%) !important;
    border: none !important;
    font-weight: 600 !important;
    font-size: 1rem !important;
    padding: 12px 24px !important;
    border-radius: 10px !important;
    transition: all 0.3s !important;
}
.primary-btn:hover {
    transform: translateY(-2px) !important;
    box-shadow: 0 8px 20px rgba(102,126,234,0.4) !important;
}
.secondary-btn {
    background: rgba(255,255,255,0.1) !important;
    border: 1px solid rgba(255,255,255,0.2) !important;
    font-weight: 500 !important;
    border-radius: 10px !important;
}

/* Example buttons */
.example-btn {
    background: rgba(102,126,234,0.2) !important;
    border: 1px solid rgba(102,126,234,0.4) !important;
    border-radius: 20px !important;
    font-size: 0.85rem !important;
    padding: 8px 16px !important;
    transition: all 0.2s !important;
}
.example-btn:hover {
    background: rgba(102,126,234,0.4) !important;
    transform: scale(1.05) !important;
}

/* Status */
.status-box input {
    background: transparent !important;
    border: none !important;
    color: #808080 !important;
    font-size: 0.8rem !important;
    text-align: center !important;
}

/* Labels */
.section-label {
    color: #667eea !important;
    font-size: 0.9rem !important;
    font-weight: 600 !important;
    margin-bottom: 8px !important;
    text-transform: uppercase !important;
    letter-spacing: 1px !important;
}
"""

# ═══════════════════════════════════════════════════════════════════════════════
# 🖥️ GRADIO UI
# ═══════════════════════════════════════════════════════════════════════════════

with gr.Blocks(title="🎙️ Türkçe Asistan", css=css, theme=gr.themes.Base()) as demo:
    
    # Header
    gr.HTML('<h1 class="header-title">🎙️ Türkçe Sesli Asistan</h1>')
    gr.HTML('<p class="header-sub">Whisper + LM Studio + Tool Calling + Cache + Memory</p>')
    
    with gr.Row():
        # ═══ LEFT PANEL - INPUT ═══
        with gr.Column(scale=1, min_width=300):
            gr.HTML('<div class="section-label">🎤 Ses veya Metin</div>')
            
            audio = gr.Audio(sources=["microphone", "upload"], type="filepath", label="", show_label=False)
            audio_btn = gr.Button("🎯 Sesi İşle", variant="primary", elem_classes=["primary-btn"])
            
            gr.HTML('<hr style="border-color: rgba(255,255,255,0.1); margin: 16px 0;">')
            
            text = gr.Textbox(lines=2, placeholder="Sorunuzu yazın...", label="", show_label=False)
            
            with gr.Row():
                text_btn = gr.Button("💬 Gönder", elem_classes=["secondary-btn"], scale=3)
                clear_btn = gr.Button("🗑️", elem_classes=["secondary-btn"], scale=1)
            
            status = gr.Textbox(value=get_status(), interactive=False, label="", show_label=False, elem_classes=["status-box"])
        
        # ═══ RIGHT PANEL - OUTPUT ═══
        with gr.Column(scale=2, min_width=400):
            gr.HTML('<div class="section-label">📝 Transkripsiyon</div>')
            trans = gr.Textbox(interactive=False, lines=2, placeholder="Ses metne dönüştürülecek...", 
                              label="", show_label=False, elem_classes=["trans-box"])
            
            gr.HTML('<div class="section-label" style="margin-top:16px;">🤖 Yanıt</div>')
            resp = gr.Textbox(interactive=False, lines=6, placeholder="Asistan yanıtı burada...", 
                             label="", show_label=False, elem_classes=["resp-box"])
            
            gr.HTML('<div class="section-label" style="margin-top:16px;">🔧 Tool</div>')
            tool = gr.HTML('<div class="tool-empty">Waiting for tool...</div>')
    
    # ═══ EXAMPLES ═══
    gr.HTML('<div class="section-label" style="text-align:center; margin-top:24px;">💡 Hızlı Örnekler</div>')
    with gr.Row():
        ex1 = gr.Button("🐍 Python nedir?", elem_classes=["example-btn"])
        ex2 = gr.Button("🔢 125 × 48", elem_classes=["example-btn"])
        ex3 = gr.Button("🕐 Saat kaç?", elem_classes=["example-btn"])
        ex4 = gr.Button("🌤️ İstanbul hava", elem_classes=["example-btn"])
        ex5 = gr.Button("📰 Güncel haberler", elem_classes=["example-btn"])
    
    # ═══ EVENTS ═══
    audio_btn.click(process_audio_ui, [audio], [trans, resp, tool, status])
    text_btn.click(process_text_ui, [text], [trans, resp, tool, status])
    text.submit(process_text_ui, [text], [trans, resp, tool, status])
    clear_btn.click(clear_all, [], [trans, resp, tool, status])
    
    ex1.click(lambda: run_example("Python nedir?"), [], [trans, resp, tool, status])
    ex2.click(lambda: run_example("125 çarpı 48 kaç eder?"), [], [trans, resp, tool, status])
    ex3.click(lambda: run_example("Saat kaç?"), [], [trans, resp, tool, status])
    ex4.click(lambda: run_example("İstanbul hava durumu nasıl?"), [], [trans, resp, tool, status])
    ex5.click(lambda: run_example("Türkiye güncel haberler"), [], [trans, resp, tool, status])

log.success("UI hazır!")

✅ UI hazır!


## 🚀 Launch

In [9]:
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://99e1ce5c5a9df11885.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
